# RSDehamba-Lite Kaggle Workflow

This notebook runs the practical one-week pipeline for the reduced-VDB `[1,2,2]` RSDehamba-style variant. It first validates the environment and dataset layout, then runs tests, a dry run, a tiny smoke-training stage, and finally gives the full-training command.

This is not an exact official RSDehamba reproduction; it is a runnable RSDehamba-inspired lightweight variant based on the public paper description.

In [ ]:
import os
from pathlib import Path
import torch

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('inputs:', os.listdir('/kaggle/input'))

## Set Project Directory

Upload this project to Kaggle as a dataset or copy it into `/kaggle/working`, then adjust `PROJECT_DIR` if needed.

In [ ]:
PROJECT_DIR = Path('/kaggle/input/rsdehamba-lite-project')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path('/kaggle/working/rsdehamba-lite-project')
print(PROJECT_DIR)
print('exists:', PROJECT_DIR.exists())

In [ ]:
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
!pip install -q -e .

## Locate SateHaze1K

Run this cell, inspect the printed folders, and update `DATA_ROOT` if Kaggle uses a different folder name.

In [ ]:
for root in Path('/kaggle/input').iterdir():
    print(root)
    for child in list(root.iterdir())[:10]:
        print('  ', child.name)

DATA_ROOT = '/kaggle/input/satehaze1k'
print('DATA_ROOT =', DATA_ROOT)

In [ ]:
!python scripts/inspect_dataset.py --data-root "$DATA_ROOT"

If the inspector reports `pairs: 0`, stop here and share the folder tree. The loader aliases need to be adjusted before training.

In [ ]:
!pytest -q
!python scripts/train.py --config configs/rsdehamba_lite_smoke.json --data-root "$DATA_ROOT" --dry-run

## Tiny Smoke Training

This run is only for pipeline validation. It uses `128x128`, `base_dim=24`, 2 epochs, and 5% of discovered pairs.

In [ ]:
!python scripts/train.py \
  --config configs/rsdehamba_lite_smoke.json \
  --data-root "$DATA_ROOT" \
  --output-dir /kaggle/working/rsdehamba_lite_outputs

In [ ]:
!python scripts/evaluate.py \
  --config configs/rsdehamba_lite_smoke.json \
  --data-root "$DATA_ROOT" \
  --checkpoint /kaggle/working/rsdehamba_lite_outputs/checkpoints/best.pt

## Full Training Command

Run this after the smoke stage works. Keep the result caveat in the report: this is a reduced-VDB variant and not an exact official reproduction.

In [ ]:
!python scripts/train.py \
  --config configs/rsdehamba_lite.json \
  --data-root "$DATA_ROOT" \
  --output-dir /kaggle/working/rsdehamba_lite_full \
  --epochs 50 \
  --batch-size 4 \
  --image-size 256

In [ ]:
!python scripts/evaluate.py \
  --config configs/rsdehamba_lite.json \
  --data-root "$DATA_ROOT" \
  --checkpoint /kaggle/working/rsdehamba_lite_full/checkpoints/best.pt